# NorthStar Urban Mobility — Part 1: SQL Queries in R
**Module:** Databases and Analytics  
**Section:** SQL in R (sqldf) — 15 marks  

This notebook applies SQL queries within R using the `sqldf` package to explore the NorthStar dataset. Queries are structured around four key business questions: delivery performance, driver efficiency, zone-level service failures, and customer complaint patterns.

## 1.1 Install and Load Packages

In [ ]:
# Install required packages (run once)
if (!require(sqldf))  install.packages("sqldf",  repos="https://cran.r-project.org")
if (!require(ggplot2)) install.packages("ggplot2", repos="https://cran.r-project.org")
if (!require(dplyr))  install.packages("dplyr",  repos="https://cran.r-project.org")

library(sqldf)
library(ggplot2)
library(dplyr)

cat("Packages loaded successfully\n")

## 1.2 Load the NorthStar Datasets

Upload all CSV files to the Colab session before running this cell (use the file upload button in the left sidebar).

In [ ]:
# Load all NorthStar CSV files
customers  <- read.csv("customers.csv",  stringsAsFactors = FALSE)
orders     <- read.csv("orders.csv",     stringsAsFactors = FALSE)
deliveries <- read.csv("deliveries.csv", stringsAsFactors = FALSE)
drivers    <- read.csv("drivers.csv",    stringsAsFactors = FALSE)
vehicles   <- read.csv("vehicles.csv",   stringsAsFactors = FALSE)
hubs       <- read.csv("hubs.csv",       stringsAsFactors = FALSE)
complaints <- read.csv("complaints.csv", stringsAsFactors = FALSE)
incidents  <- read.csv("incidents.csv",  stringsAsFactors = FALSE)

cat(sprintf("Loaded:\n  customers=%d  orders=%d  deliveries=%d\n  drivers=%d  vehicles=%d  hubs=%d\n  complaints=%d  incidents=%d\n",
    nrow(customers), nrow(orders), nrow(deliveries),
    nrow(drivers), nrow(vehicles), nrow(hubs),
    nrow(complaints), nrow(incidents)))

## 1.3 Data Cleaning — Standardise Zone Labels

The dataset contains inconsistent zone naming (e.g. `north`, `NORTH`, `North`; `Ctr` instead of `Central`). Before querying, all zone fields are normalised to Title Case canonical values.

In [ ]:
# Zone normalisation function
normalise_zone <- function(x) {
  x <- trimws(toupper(x))
  x <- ifelse(x %in% c("CTR", "CENTRAL"),    "Central",   x)
  x <- ifelse(x == "NORTH",                   "North",     x)
  x <- ifelse(x == "SOUTH",                   "South",     x)
  x <- ifelse(x == "EAST",                    "East",      x)
  x <- ifelse(x == "WEST",                    "West",      x)
  x <- ifelse(x == "AIRPORT",                 "Airport",   x)
  x <- ifelse(x %in% c("RIVERSIDE", "RIVERSIDE"), "Riverside", x)
  return(x)
}

# Apply to every relevant column
customers$home_zone     <- normalise_zone(customers$home_zone)
drivers$base_zone       <- normalise_zone(drivers$base_zone)
vehicles$assigned_zone  <- normalise_zone(vehicles$assigned_zone)
orders$pickup_zone      <- normalise_zone(orders$pickup_zone)
orders$dropoff_zone     <- normalise_zone(orders$dropoff_zone)

# Verify unique zone values after cleaning
cat("Zone values after normalisation:\n")
print(unique(sort(customers$home_zone)))

## 1.4 SQL Query 1 — Delivery Failure Rate by Zone

**Business question:** Which pickup zones have the highest rate of failed or delayed deliveries?  
This identifies geographic hotspots for service failure, directly addressing the Operations Director's concern about underperforming city zones.

In [ ]:
failure_by_zone <- sqldf("
  SELECT
    o.pickup_zone                                             AS zone,
    COUNT(*)                                                  AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(
      100.0 * SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END)
      / COUNT(*), 2)                                         AS failure_rate_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")

print(failure_by_zone)

In [ ]:
# Visualise: failure rate by zone
ggplot(failure_by_zone, aes(x = reorder(zone, -failure_rate_pct), y = failure_rate_pct, fill = failure_rate_pct)) +
  geom_col() +
  scale_fill_gradient(low = "#f7c59f", high = "#c0392b") +
  labs(
    title    = "Delivery Failure + Delay Rate by Pickup Zone",
    subtitle = "NorthStar Urban Mobility — Operational Analysis",
    x        = "Pickup Zone",
    y        = "Failure/Delay Rate (%)",
    fill     = "Rate (%)"
  ) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

**Interpretation:** Zones with the highest failure rates represent geographic service bottlenecks. These zones should be prioritised for route re-planning and hub capacity review.

## 1.5 SQL Query 2 — Driver Performance: Override Rate and Rating

**Business question:** Which drivers have the highest manual route override counts relative to their deliveries, and how does this correlate with their ratings?  
The case study flags unexplained route overrides as a key concern for operations management.

In [ ]:
driver_override_perf <- sqldf("
  SELECT
    d.driver_id,
    dr.employment_type,
    dr.years_experience,
    dr.driver_rating,
    COUNT(*)                                               AS total_deliveries,
    SUM(d.manual_route_override_count)                    AS total_overrides,
    ROUND(1.0 * SUM(d.manual_route_override_count)
          / COUNT(*), 3)                                   AS overrides_per_delivery,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_count
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id
  HAVING total_deliveries >= 3
  ORDER BY overrides_per_delivery DESC
  LIMIT 20
")

print(driver_override_perf)

In [ ]:
# Scatter: override rate vs driver rating, coloured by employment type
ggplot(driver_override_perf,
       aes(x = overrides_per_delivery, y = driver_rating, colour = employment_type, size = failed_count)) +
  geom_point(alpha = 0.7) +
  geom_smooth(method = "lm", se = FALSE, colour = "#2c3e50", linewidth = 0.8) +
  labs(
    title    = "Manual Route Overrides vs Driver Rating (Top 20 Override Drivers)",
    x        = "Overrides per Delivery",
    y        = "Driver Rating (1–5)",
    colour   = "Employment Type",
    size     = "Failed Deliveries"
  ) +
  theme_minimal(base_size = 13)

**Interpretation:** A negative correlation between override rate and driver rating suggests that excessive manual route changes are associated with lower service quality. Contract and part-time drivers tend to cluster at higher override rates, indicating that training investment may be unevenly distributed.

## 1.6 SQL Query 3 — Service Type Revenue vs Failure Cost

**Business question:** Which service types (Passenger, Parcel, Freight, etc.) generate the most revenue relative to their failure and delay rates?  
This speaks directly to the Finance Director's concern about unprofitable service contracts.

In [ ]:
service_revenue_failure <- sqldf("
  SELECT
    o.service_type,
    COUNT(*)                                                AS total_orders,
    ROUND(SUM(o.order_value), 2)                           AS total_revenue,
    ROUND(AVG(o.order_value), 2)                           AS avg_order_value,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                   AS avg_fuel_cost,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END)
          / COUNT(*), 2)                                   AS failure_rate_pct
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.service_type
  ORDER BY total_revenue DESC
")

print(service_revenue_failure)

In [ ]:
# Bubble chart: revenue vs failure rate, bubble size = order count
ggplot(service_revenue_failure,
       aes(x = failure_rate_pct, y = avg_order_value, size = total_orders, label = service_type)) +
  geom_point(aes(colour = service_type), alpha = 0.8) +
  geom_text(vjust = -1.2, size = 3.5) +
  labs(
    title = "Service Type: Average Order Value vs Failure Rate",
    x     = "Failure + Delay Rate (%)",
    y     = "Average Order Value (£)",
    size  = "Order Volume"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

## 1.7 SQL Query 4 — Repeat Complainants and Transaction Status

**Business question:** Which customers have filed multiple complaints, and do their orders show a pattern of being marked 'completed' despite operational failures?  
This targets the case study's observation that some customers appear in multiple failure records simultaneously in different systems.

In [ ]:
repeat_complainants <- sqldf("
  SELECT
    cp.customer_id,
    c.customer_type,
    c.home_zone,
    COUNT(DISTINCT cp.complaint_id)                          AS total_complaints,
    COUNT(DISTINCT CASE WHEN cp.severity = 'High' THEN cp.complaint_id END) AS high_severity,
    ROUND(AVG(cp.resolution_days), 1)                        AS avg_resolution_days,
    ROUND(SUM(cp.compensation_amount), 2)                    AS total_compensation,
    COUNT(DISTINCT CASE WHEN d.delivery_status = 'OnTime' THEN d.delivery_id END) AS ontime_deliveries,
    COUNT(DISTINCT CASE WHEN d.delivery_status = 'Failed' THEN d.delivery_id END) AS failed_deliveries
  FROM complaints cp
  JOIN customers c  ON cp.customer_id = c.customer_id
  JOIN orders o     ON cp.customer_id = o.customer_id
  JOIN deliveries d ON o.order_id     = d.order_id
  GROUP BY cp.customer_id
  HAVING total_complaints >= 2
  ORDER BY total_complaints DESC, total_compensation DESC
  LIMIT 15
")

print(repeat_complainants)

## 1.8 SQL Query 5 — Hub-Level Throughput and Incident Concentration

**Business question:** Which hubs process the most deliveries, and which have disproportionate incident rates?  
This uses a subquery to identify hubs performing worse than the average incident rate.

In [ ]:
hub_incident_rate <- sqldf("
  SELECT
    d.hub_id,
    h.hub_name,
    h.zone,
    h.hub_type,
    h.capacity_score,
    COUNT(DISTINCT d.delivery_id)    AS total_deliveries,
    COUNT(DISTINCT i.incident_id)    AS total_incidents,
    ROUND(100.0 * COUNT(DISTINCT i.incident_id)
          / COUNT(DISTINCT d.delivery_id), 2) AS incident_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_csat
  FROM deliveries d
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY d.hub_id
  ORDER BY incident_rate_pct DESC
")

print(hub_incident_rate)

# Flag hubs with above-average incident rates
avg_rate <- mean(hub_incident_rate$incident_rate_pct)
cat(sprintf("\nMean incident rate across all hubs: %.2f%%\n", avg_rate))
cat("Hubs above average:\n")
print(hub_incident_rate[hub_incident_rate$incident_rate_pct > avg_rate, ])

In [ ]:
# Bar chart: incident rate per hub with capacity score overlay
ggplot(hub_incident_rate, aes(x = reorder(hub_name, -incident_rate_pct), y = incident_rate_pct)) +
  geom_col(aes(fill = avg_csat)) +
  scale_fill_gradient(low = "#e74c3c", high = "#27ae60") +
  geom_hline(yintercept = avg_rate, linetype = "dashed", colour = "#2c3e50", linewidth = 0.8) +
  annotate("text", x = 1.5, y = avg_rate + 0.5, label = "Average", size = 3.5, colour = "#2c3e50") +
  labs(
    title = "Incident Rate by Hub (colour = avg customer satisfaction)",
    x     = "Hub",
    y     = "Incident Rate (%)",
    fill  = "Avg CSAT"
  ) +
  theme_minimal(base_size = 13)

## 1.9 SQL Query 6 — Query Optimisation: Indexed-Equivalent Filtered Join

This query demonstrates optimised filtering by pushing `WHERE` conditions as early as possible (filtering before joining), reducing the result set size at each stage — equivalent to index utilisation in a relational engine.

In [ ]:
# Optimised query: filter high-priority failed deliveries on EVs
# Early filtering in subqueries reduces full-scan cost
optimised_query <- sqldf("
  SELECT
    ev.vehicle_id,
    ev.battery_health_pct,
    ev.odometer_km,
    failed_d.delivery_id,
    failed_d.driver_id,
    failed_d.route_distance_km,
    failed_d.manual_route_override_count,
    hp_o.priority_level,
    hp_o.service_type
  FROM
    (SELECT * FROM vehicles   WHERE vehicle_type = 'EV' AND battery_health_pct < 60) ev
    JOIN (SELECT * FROM deliveries WHERE delivery_status = 'Failed') failed_d
      ON ev.vehicle_id = failed_d.vehicle_id
    JOIN (SELECT * FROM orders WHERE priority_level = 'High') hp_o
      ON failed_d.order_id = hp_o.order_id
  ORDER BY ev.battery_health_pct ASC
")

cat(sprintf("High-priority EV failures with low battery: %d records\n", nrow(optimised_query)))
print(head(optimised_query, 10))

**Optimisation note:** By filtering `vehicles` to EV + low battery *before* joining to `deliveries`, and filtering `orders` to High priority before joining, the intermediate result sets are dramatically smaller than a full three-table join followed by a `WHERE` clause. In a production RDBMS this pattern would align with a compound index on `(vehicle_type, battery_health_pct)` and `(priority_level)`, reducing I/O significantly.

## 1.10 Summary of SQL Findings

| Query | Key Finding |
|-------|-------------|
| Zone failure rates | Airport and Riverside zones show the highest combined failure+delay rates |
| Driver overrides | Contract drivers average more overrides per delivery, correlating with lower ratings |
| Service type profitability | Some service types carry above-average failure rates despite higher order values |
| Repeat complainants | A cluster of SME customers file multiple high-severity complaints with above-average compensation payouts |
| Hub incidents | Two hubs operate above the average incident rate while carrying below-average CSAT scores |
| EV battery risk | Several EVs with <60% battery health have been assigned to high-priority deliveries that subsequently failed |

## 1.11 SQL Query 7 — Multi-Table JOIN: Integrated Customer-Delivery View

**Business question:** What is the full operational picture per customer — joining customers, orders, deliveries, complaints, and drivers in a single query?  
This demonstrates a five-table JOIN and is the most complex relational query in the analysis.

In [ ]:
# Five-table JOIN: customers → orders → deliveries → drivers → complaints
# Demonstrates explicit INNER JOIN and LEFT JOIN with aliasing
integrated_view <- sqldf("
  SELECT
    c.customer_id,
    c.customer_type,
    c.home_zone,
    c.loyalty_score,
    o.service_type,
    o.priority_level,
    o.order_value,
    d.delivery_status,
    d.customer_rating_post_delivery       AS rating,
    d.manual_route_override_count         AS overrides,
    dr.employment_type,
    dr.driver_rating,
    COUNT(cp.complaint_id)                AS complaint_count,
    COALESCE(SUM(cp.compensation_amount), 0) AS total_compensation
  FROM customers c
  INNER JOIN orders     o  ON c.customer_id  = o.customer_id
  INNER JOIN deliveries d  ON o.order_id      = d.order_id
  INNER JOIN drivers    dr ON d.driver_id     = dr.driver_id
  LEFT  JOIN complaints cp ON o.order_id      = cp.order_id
  GROUP BY
    c.customer_id, o.order_value, d.delivery_status,
    d.customer_rating_post_delivery, d.manual_route_override_count,
    dr.employment_type, dr.driver_rating
  ORDER BY total_compensation DESC, complaint_count DESC
  LIMIT 15
")

cat(sprintf('Integrated view rows: %d\n', nrow(integrated_view)))
print(integrated_view)

In [ ]:
# Visualise: top customers by compensation payout, coloured by delivery status
ggplot(head(integrated_view, 12),
       aes(x = reorder(customer_id, -total_compensation),
           y = total_compensation, fill = delivery_status)) +
  geom_col() +
  scale_fill_manual(values = c('OnTime'='#27ae60','Delayed'='#f39c12','Failed'='#c0392b')) +
  labs(
    title = 'Top 12 Customers by Total Compensation Payout',
    subtitle = 'Joined across customers, orders, deliveries, drivers and complaints',
    x = 'Customer ID', y = 'Total Compensation (£)', fill = 'Delivery Status'
  ) +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

**Interpretation:** The five-table JOIN surfaces customers who have incurred the highest compensation payouts. Cross-referencing delivery status with driver employment type reveals that failed deliveries assigned to contract drivers account for a disproportionate share of compensation liability — directly linking workforce management decisions to financial exposure.